<a href="https://colab.research.google.com/github/lPragathi/project/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required packages
!pip install -q langgraph langchain-core langchain-openai

import os
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# ==========================================
# RETRIEVE ACTIVE API KEY FROM LOCAL STORAGE
# ==========================================
# Accessing the working credentials in your environment dynamically
OPENROUTER_API_KEY = globals().get("API_KEY", os.environ.get("API_KEY", ""))

# 2. Initialize the Three LLMs via OpenRouter
llm_coder = ChatOpenAI(
    model="deepseek/deepseek-chat",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

llm_checker = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

llm_corrector = ChatOpenAI(
    model="google/gemini-2.5-pro",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# 3. Define State Structure with Initializers
class GraphState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    generated_code: str
    error_report: str
    is_error_free: bool
    iteration_count: int

# 4. Define Nodes
def coder_node(state: GraphState):
    print("\n--- [LLM 1: CODER] Writing code... ---")
    history = state.get("messages", [])
    prompt = HumanMessage(content="Write clean, fully executable Python code based on the user's request. Return ONLY the code inside a standard markdown code block.")
    response = llm_coder.invoke(history + [prompt])

    current_count = state.get("iteration_count", 0) + 1
    return {
        "generated_code": response.content,
        "iteration_count": current_count
    }

def checker_node(state: GraphState):
    print("\n--- [LLM 2: CHECKER] Inspecting code for errors... ---")
    code = state.get("generated_code", "")
    prompt = HumanMessage(content=f"Review the following code for syntax errors, logical bugs, or missing imports:\n\n{code}\n\n"
                                  "If there are any errors or bugs, start your response with 'STATUS: ERROR' followed by the details. "
                                  "If the code is clean and correct, start your response with 'STATUS: CLEAN'.")

    response = llm_checker.invoke([prompt])
    content = response.content

    is_clean = "STATUS: CLEAN" in content.upper()
    return {
        "error_report": content,
        "is_error_free": is_clean
    }

def corrector_node(state: GraphState):
    print("\n--- [LLM 3: CORRECTOR] Fixing errors... ---")
    code = state.get("generated_code", "")
    report = state.get("error_report", "")
    prompt = HumanMessage(content=f"The previous code had issues:\n{code}\n\n"
                                  f"Reviewer Feedback:\n{report}\n\n"
                                  "Rewrite the code completely to fix all identified bugs. Ensure it is fully functional and wrapped in a markdown code block.")

    response = llm_corrector.invoke([prompt])
    return {
        "generated_code": response.content,
        "is_error_free": True
    }

# 5. Define Routing Logic
def route_decision(state: GraphState) -> str:
    print("\n--- ROUTING EVALUATION ---")
    if state.get("is_error_free", False) or state.get("iteration_count", 0) >= 3:
        return "end"
    return "corrector"

# 6. Build and Compile Graph
workflow = StateGraph(GraphState)

workflow.add_node("coder", coder_node)
workflow.add_node("checker", checker_node)
workflow.add_node("corrector", corrector_node)

workflow.set_entry_point("coder")
workflow.add_edge("coder", "checker")
workflow.add_conditional_edges(
    "checker",
    route_decision,
    {
        "end": END,
        "corrector": "corrector"
    }
)
workflow.add_edge("corrector", "checker")

app = workflow.compile()

# 7. Execution
if __name__ == "__main__":
    initial_state = {
        "messages": [HumanMessage(content="Write a Python function to safely fetch data from a public REST API with error handling and retries.")],
        "generated_code": "",
        "error_report": "",
        "is_error_free": False,
        "iteration_count": 0
    }

    result = app.invoke(initial_state)
    print("\n==============================")
    print("FINAL VERIFIED CODE OUTPUT")
    print("==============================")
    print(result.get("generated_code", "No code generated"))


--- [LLM 1: CODER] Writing code... ---

--- [LLM 2: CHECKER] Inspecting code for errors... ---

--- ROUTING EVALUATION ---

FINAL VERIFIED CODE OUTPUT
```python
import requests
import time

def fetch_data_from_api(url, max_retries=3, delay=2):
    """
    Safely fetch data from a public REST API with error handling and retries.

    :param url: The URL of the REST API endpoint.
    :param max_retries: Maximum number of retries in case of failure.
    :param delay: Delay in seconds between retries.
    :return: The JSON response from the API or None if the request fails.
    """
    for attempt in range(max_retries):
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
            return response.json()  # Return the JSON response if successful
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries

In [ ]:
import os
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# 1. Retrieve the Active API Key
OPENROUTER_API_KEY = globals().get("API_KEY", os.environ.get("API_KEY", ""))

# 2. Initialize our 3 LLMs
llm_coder = ChatOpenAI(
    model="deepseek/deepseek-chat",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

llm_checker = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

llm_corrector = ChatOpenAI(
    model="google/gemini-2.5-pro",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# 3. Define Graph State
class ResilientState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    generated_code: str
    error_report: str
    is_error_free: bool
    iteration_count: int

# 4. Define Nodes with Robust Error Handling & Fallbacks
def coder_node(state: ResilientState):
    print("\n--- [LLM 1: CODER] Attempting generation... ---")
    history = state.get("messages", [])
    prompt = HumanMessage(content="Write clean, fully executable Python code based on the user's request. Return ONLY the code inside a standard markdown code block.")

    try:
        response = llm_coder.invoke(history + [prompt])
        generated_code = response.content
    except Exception as e:
        print(f"⚠️ LLM 1 (Coder) Failed: {e}. Passing control to LLM 3 fallback node.")
        generated_code = ""  # Leave empty to trigger corrector fallback node

    return {
        "generated_code": generated_code,
        "iteration_count": state.get("iteration_count", 0) + 1
    }

def checker_node(state: ResilientState):
    print("\n--- [LLM 2: CHECKER] Checking code validity... ---")
    code = state.get("generated_code", "")

    # If Coder completely failed to return code, mark as error to force Corrector fallback
    if not code.strip():
        return {
            "error_report": "STATUS: ERROR - Primary Coder node returned empty code.",
            "is_error_free": False
        }

    prompt = HumanMessage(content=f"Review the following code for syntax errors, logical bugs, or missing imports:\n\n{code}\n\n"
                                  "If there are any errors, start your response with 'STATUS: ERROR'. "
                                  "If the code is clean and correct, start your response with 'STATUS: CLEAN'.")
    try:
        response = llm_checker.invoke([prompt])
        content = response.content
        is_clean = "STATUS: CLEAN" in content.upper()
    except Exception as e:
        print(f"⚠️ LLM 2 (Checker) Failed: {e}. Activating corrector fallback node.")
        content = f"STATUS: ERROR - Checker execution failed with: {e}"
        is_clean = False

    return {
        "error_report": content,
        "is_error_free": is_clean
    }

def corrector_fallback_node(state: ResilientState):
    print("\n--- [LLM 3: CORRECTOR / FALLBACK] Recovering from error/failure... ---")
    history = state.get("messages", [])
    code = state.get("generated_code", "")
    report = state.get("error_report", "")

    # If coder failed completely (empty code), LLM 3 takes over primary role
    if not code.strip():
        prompt = HumanMessage(content="The primary Coder failed to run. You are the fallback generator. Write clean, fully executable Python code based on the user's request inside a markdown block.")
    else:
        prompt = HumanMessage(content=f"The previous code had issues:\n{code}\n\n"
                                      f"Reviewer Feedback:\n{report}\n\n"
                                      "Rewrite the code completely to fix all identified bugs and wrap it in a markdown code block.")

    response = llm_corrector.invoke(history + [prompt])
    return {
        "generated_code": response.content,
        "is_error_free": True  # Set to true to allow checking of the new code
    }

# 5. Routing Logic
def route_decision(state: ResilientState) -> str:
    print("\n--- ROUTING EVALUATION ---")
    if state.get("is_error_free", False) or state.get("iteration_count", 0) >= 3:
        return "end"
    return "corrector"

# 6. Build the Resilient Graph Flow
workflow = StateGraph(ResilientState)

workflow.add_node("coder", coder_node)
workflow.add_node("checker", checker_node)
workflow.add_node("corrector", corrector_fallback_node)

workflow.set_entry_point("coder")
workflow.add_edge("coder", "checker")
workflow.add_conditional_edges(
    "checker",
    route_decision,
    {
        "end": END,
        "corrector": "corrector"
    }
)
workflow.add_edge("corrector", "checker")

app = workflow.compile()

# 7. Demo Execution with Fallbacks Active
if __name__ == "__main__":
    initial_state = {
        "messages": [HumanMessage(content="Write a Python script to verify if a string is a palindrome.")],
        "generated_code": "",
        "error_report": "",
        "is_error_free": False,
        "iteration_count": 0
    }

    result = app.invoke(initial_state)
    print("\n==============================")
    print("FINAL RESILIENT CODE OUTPUT")
    print("==============================")
    print(result.get("generated_code", "No code generated"))


--- [LLM 1: CODER] Attempting generation... ---

--- [LLM 2: CHECKER] Checking code validity... ---

--- ROUTING EVALUATION ---

FINAL RESILIENT CODE OUTPUT
```python
def is_palindrome(s):
    # Remove any non-alphanumeric characters and convert to lowercase
    cleaned_string = ''.join(char.lower() for char in s if char.isalnum())
    # Check if the cleaned string is equal to its reverse
    return cleaned_string == cleaned_string[::-1]

# Example usage
input_string = "A man, a plan, a canal, Panama"
if is_palindrome(input_string):
    print(f'"{input_string}" is a palindrome.')
else:
    print(f'"{input_string}" is not a palindrome.')
```
